<div style="display: flex; justify-content: space-between; align-items: flex-start; border-bottom: 2px solid #555555; padding-bottom: 15px; margin-bottom: -8px;">
    <div style="width: 50%;">
        <h2>
            <span style="color: #B30033;">▍</span>TRABAJO DE FIN DE MÁSTER
        </h2>
        <h1 style="margin-top: -10px;">
            Creación de un framework para recomendar las personas más adecuadas para trabajar en un determinado proyecto de ingeniería del software
        </h1>
    </div>
    <div style="width: 50%; text-align: right;">
        <div style="display: flex; justify-content: space-between; align-items: flex-start; margin-top: 30px;">
            <div style="width: 20%;"></div>
            <div style="width: 80%; border-left: 2px solid #555555; padding-left: 20px;">
                <div style="margin-bottom: 20px;">
                    <p style="margin: 0; font-size: 1.4em; font-weight: bold;">
                        Profesores
                    </p>
                </div>
                <div style="margin-top: 8px; text-align: right;">
                    <span style="display: block; font-size: 1em; color: #D0D0D0;">
                        Aurora Vizcaíno Barceló
                    </span>
                    <span style="display: block; font-size: 1em; color: #D0D0D0; margin-top: 5px;">
                        Julio Alberto López Gómez
                    </span>
                </div>
            </div>
        </div>
    </div>
</div>

<div style="margin-top: 30px; padding: 20px; background-color: rgba(179, 0, 51, 0.05); border-left: 4px solid #B30033; border-radius: 4px;">
    <h3 style="margin-top: 0; color: #B30033;">🎯 Objetivo de esta Libreta (Fase 3: Minería de Reglas de Asociación)</h3>
    <p style="font-size: 1.1em; line-height: 1.6; color: #E0E0E0;">
        El objetivo de esta tercera libreta es <strong>descubrir patrones ocultos y relaciones entre los distintos datos ya procesados de la matriz, así como preferencias y características de la personalidad</strong> de los diferentes candidatos, utilizando algoritmos de aprendizaje no supervisados basados en reglas.
    </p>
    <ul style="font-size: 1.1em; line-height: 1.6; color: #D0D0D0; margin-bottom: 0;">
        <li><strong>Carga y optimización:</strong> Se importa la matriz transaccional generada en la libreta 2 y se convierte a booleana para así ser más eficiente en memoria.</li>
        <li><strong>Extracción los Itemsets Frecuentes:</strong> A lo largo de la libreta vamos a aplicar 3 algoritmos principales y a compararlos entre ellos. Estos algoritmos, todos de minería de reglas de asociación son: <strong>Apriori</strong>, <strong>FP-Growth</strong> y <strong>Eclat</strong>. Para elegir cual conviene más evaluaremos tanto la equivalencia de los resultados obtenidos como su eficiencia computacional y tiempos.</li>
        <li><strong>Generación de Reglas:</strong> Creamos reglas direccionales (Antecedente → Consecuente) basadas en métricas de probabilidad condicional, como son la (<em>Confianza</em>) y la fuerza de asociación (<em>Lift</em>).</li>
        <li><strong>Filtrado por Relevancia:</strong> Posteriormente vamos a depurar reglas que consideramos innecesarias, excluyendo aquellas que en su mayoría tengan valores nulos y aquellas circulares respecto a la propia construcción de los proxies de personalidad (Big Five).</li>
        <li><strong>Exportación del modelo:</strong> Guardaremos las reglas finales como "Base de Conocimiento" para asi cruzarla más tardes con posteriormente con las reglas expertas de Aurora.</li>
    </ul>
</div>
<br/>

<div style="border-bottom: 2px solid #555555; padding-bottom: 25px; margin-bottom: 10px">
    <div style="display: flex; align-items: center; margin-bottom: 10px;">
        <span style="color: #B30033; font-size: 1.5em; margin-right: 10px;">▍</span>
        <h3 style="margin: 0; font-size: 1.4em; font-weight: bold">
            Realizado por Miguel Ángel Sánchez-Medina Camacho <span style="font-size: 0.7em; font-style: italic; font-weight: normal;"></span>
        </h3>
    </div>
</div>

### 3.1. Configuración del Entorno y carga de la Matriz Transaccional

Primero como siempre importamos todas las librerias necesarias. Para esta parte del proyecto, haremos uso de las librerias siguientes:

* `mlxtend` para la minería de reglas, de ella importamos apriori y fpgrowth.
* Después importamos `pyECLAT` para posteriormente evaluar este algoritmo.
* Cargamos la matriz transaccional y la convertimos a booleana, para que tenga unicamente valores `True`/`False`. Esto lo hacemos por que es el formato óptimo que requiere `mlxtend` para un procesamiento rápido y de bajo consumo de memoria.

In [1]:
import os
import sys
import warnings
import time

sys.path.append(os.path.abspath(os.path.join('../..')))
from src.soft_skills import config

import pandas as pd
from mlxtend.frequent_patterns import apriori,fpgrowth, association_rules
from IPython.display import display
from pyECLAT import ECLAT

warnings.filterwarnings("ignore")

df_matriz = pd.read_csv(config.MATRIZ_TRANSACCIONAL_PATH, index_col='CandidateId')
df_matriz = df_matriz.fillna(0).astype(bool)

print(f"Dimensiones matriz: {df_matriz.shape}")
display(df_matriz.head(3))

Dimensiones matriz: (88778, 205)


,WorkWeekHrs_is_missing,CodeRevHrs_is_missing,WorkChallenge_Distracting work environment,WorkChallenge_Meetings,WorkChallenge_Being tasked with non-development work,WorkChallenge_Not enough people for the workload,WorkChallenge_Lack of support from management,WorkChallenge_Inadequate access to necessary tools,WorkChallenge_Toxic work environment,WorkChallenge_Time spent commuting,...,Apertura_Nivel_Bajo,Apertura_Nivel_Medio,WorkWeekHrs_Bin_35-45h,WorkWeekHrs_Bin_45-55h,WorkWeekHrs_Bin_<35h,WorkWeekHrs_Bin_>55h,CodeRevHrs_Bin_0-2h,CodeRevHrs_Bin_0h (No revisa nada de código),CodeRevHrs_Bin_2-5h,CodeRevHrs_Bin_>5h
CandidateId,,,,,,,,,,,,,,,,,,,,,
1,True,True,False,False,False,False,False,False,False,False,...,False,True,False,False,False,False,False,False,False,False
2,True,True,False,False,False,False,False,False,False,False,...,False,True,False,False,False,False,False,False,False,False
3,False,True,True,False,False,False,True,True,False,False,...,False,True,True,False,False,False,False,False,False,False


### 3.2. Generación de Itemsets Frecuentes — 1er algoritmo --> Apriori

Propuesto por Agrawal y Srikant en 1994, **Apriori** es uno de los primeros algortimos desarrollados para identificar itemsets frecuentes en bases de datos con transacciones, y posteriormente, conversión a reglas de asociación. Este algoritmo consta de 2 etapas: primero identifica todos los itemsets que ocurren con una frecuencia por encima de un determinado límite y posteriormente a partir de ahí construye combinaciones cada vez más grandes (2 ítems, 3 ítems...), descartando en cada paso cualquier combinación con un subconjunto no frecuente (*downward closure*).

**Conceptos clave utilizados en esta libreta:**
- **Soporte** (`support`): Es el nº de transacciones que contienen el itemset dividido entre el total de transacciones. En nuestro caso, la proporción de candidatos en los que aparece un itemset. El valor mínimo configurado es un min_support de 0.15, el cual significa que la combinación aparece en al menos el 15% de los 88.778 candidatos (~13.300 personas). En un principio el valor iba a ser menor, pero se ha ido ajustando ya que al ejecutar fallaba por problemas de memoria.
- **Confianza** (`confidence`): Podemos definirlo como la probabilidad de que se cumpla el consecuente de una regla, una vez que se cumple el antecedente. Es una probabilidad condicional, no simétrica.
 - **Lift**: Cuánto más frecuente es la combinación de antecedente y consecuente de lo que cabría esperar si fueran independientes. Un lift de 1 indica independencia, pero si ese valor está por encima de 1 tiene asociación positiva.

**Justificación de los parámetros elegidos:**
- `min_support=0.15`: Primero se probaron varios valores, como por ejemplo `min_support=0.05` (el cual es más permisivo), pero con este valor el proceso agotó la memoria disponible del equipo y forzó el cierre de vscode. Esto se produce porque el número de combinaciones candidatas crece de forma exponencial con umbrales bajos. Para evitarlo, se cambió por un umbral más conservador.
- `max_len=4`: Con este parámetro limitamos el tamaño máximo de los itemsets a 4 ítems combinados, lo que ya sería suficiente para una primera exploración interpretable y otra medida adicional de control de memoria.
- `low_memory=True`: Activamos este modo de `mlxtend`, el cual tiene como objetivo que evalúa los itemsets candidatos por lotes en lugar de materializar toda la matriz de combinaciones en memoria al mismo tiempo. También se cambió por el problema de memoria comentado anteriormente.


In [2]:
itemsets_frecuentes = apriori(
    df_matriz,
    min_support=0.15,
    use_colnames=True,
    max_len=4,
    low_memory=True,
    verbose=1
)

pd.set_option('display.max_colwidth', None)
display(itemsets_frecuentes.sort_values('support',ascending=False).head(20))

Processing 3045 combinations | Sampling itemset size 4


,support,itemsets
18,0.802643,frozenset({Hobbyist_Yes})
22,0.725461,frozenset({Employment_Employed full-time})
65,0.705817,frozenset({WelcomeChange_Just as welcome now as I felt last year})
54,0.618824,frozenset({BetterLife_Yes})
67,0.605803,frozenset({WorkRemote_Bucket_Presencial / Bajo remoto (<30%)})
58,0.595418,frozenset({Extraversion_In real life (in person)})
256,0.572022,"frozenset({Employment_Employed full-time, Hobbyist_Yes})"
286,0.564464,"frozenset({Hobbyist_Yes, WelcomeChange_Just as welcome now as I felt last year})"
354,0.558832,"frozenset({Employment_Employed full-time, WorkRemote_Bucket_Presencial / Bajo remoto (<30%)})"
48,0.545450,"frozenset({CodeRev_Yes, because I see value in code review})"


Una vez tenemos el listado con el top 20, vemos que en base al número de soporte, los itemsets individuales dominan la parte alta del ranking, mientras que cualquier combinación de 2 o más ítems solo puede ser tan frecuente como el menos común de sus componentes.

* Destacan `Hobbyist_Yes` (con un 80,3%) Nos dice que la inmensa mayoría de los candidatos programa también como afición, no solo en su empleo. Segundo está `Employment_Employed full-time` (72,5%, señala los empleados que están a tiempo completo) y `WelcomeChange_Just as welcome...` terceri (70,6%, sentido de pertenencia estable a la comunidad)

* Un dato relevante para nuestro proyecto es que `JobFactors_Languages, frameworks, and other technologies I'd be working with` (48,4%) es con diferencia el factor de decisión de empleo más citado entre las respuestas multivalor de `JobFactors` (algo que también veíamos en libretas anteriores). Esto nos confirma que el stack tecnológico sigue siendo el criterio prioritario de los candidatos a la hora de elegir un proyecto, algo muy relevante para el matching final.

### 3.3. Filtrado de reglas: estructura de nulos

Antes de seguir interpretando las reglas obtenidas, vamos a depurarlas y evitar que no queden ninguna que nos afecte las válidas y que no aporten valor alguno. Primero, vamos a ver la **estructura de los valores nulos**.

En la libreta 1 correspondiente al EDA, vimos que habían muchos datos en los proxies de personalidad que tenían mucha correlación con la ausencia de datos en variables laborales como `WorkRemote`, `WorkWeekHrs`, `CodeRevHrs`. En ellas se identifican al mismo subgrupo de candidatos que no están empleados, como pueden ser estudiantes o profesionales desempleados. También pueden contener valores de personas que no quisieron responder, pero para nosotros todos son aquellos que no tienen trabajo.

Sin embargo, aquí hay algunas de esas reglas que deben ser revisadas ya que pueden tener un nivel de confianza muy alto (p.ej  **confianza = 1.000**) que afecten a los resultados de todo el conjunto de reglas. Otro problema a tener en cuenta es que las reglas expertas no contemplan la opción de que exista "ausencia de datos", por lo que sus condiciones se basan en niveles reales de los rasgos (neuroticismo alto/bajo), no en si el dato se conoce o no. Dejar estas reglas mezcladas con el resto sesgaría cualquier comparación posterior no sería buena praxis ya que ocuparían las primeras posiciones por lift sin tener ningún equivalente contra el que contrastarse.

Todo esto lo trataremos más adelante en la libreta

In [3]:
reglas_asociacion = association_rules(itemsets_frecuentes, metric="confidence", min_threshold=0.6)

reglas_asociacion = reglas_asociacion.sort_values('lift', ascending=False)
print(f"Reglas de asociacion obtenidas: {len(reglas_asociacion)}")
display(reglas_asociacion[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(20))

Reglas de asociacion obtenidas: 5324


,antecedents,consequents,support,confidence,lift
2493,"frozenset({WorkRemote_Bucket_Desconocido (Sin datos), Hobbyist_Yes, WorkWeekHrs_is_missing})",frozenset({Neuroticismo_Nivel_Desconocido}),0.154385,0.881471,4.912138
2504,frozenset({Neuroticismo_Nivel_Desconocido}),"frozenset({WorkRemote_Bucket_Desconocido (Sin datos), Hobbyist_Yes, WorkWeekHrs_is_missing})",0.154385,0.860335,4.912138
2506,"frozenset({WorkRemote_Bucket_Desconocido (Sin datos), Hobbyist_Yes, CodeRevHrs_is_missing})",frozenset({Neuroticismo_Nivel_Desconocido}),0.154385,0.878928,4.897963
2515,frozenset({Neuroticismo_Nivel_Desconocido}),"frozenset({WorkRemote_Bucket_Desconocido (Sin datos), Hobbyist_Yes, CodeRevHrs_is_missing})",0.154385,0.860335,4.897963
2481,"frozenset({WorkRemote_Bucket_Desconocido (Sin datos), WorkWeekHrs_is_missing, CodeRevHrs_is_missing})",frozenset({Neuroticismo_Nivel_Desconocido}),0.179448,0.875666,4.879789
2492,frozenset({Neuroticismo_Nivel_Desconocido}),"frozenset({WorkRemote_Bucket_Desconocido (Sin datos), WorkWeekHrs_is_missing, CodeRevHrs_is_missing})",0.179448,1.000000,4.879789
2484,"frozenset({WorkRemote_Bucket_Desconocido (Sin datos), WorkWeekHrs_is_missing})","frozenset({Neuroticismo_Nivel_Desconocido, CodeRevHrs_is_missing})",0.179448,0.872597,4.862683
2501,"frozenset({Hobbyist_Yes, Neuroticismo_Nivel_Desconocido})","frozenset({WorkRemote_Bucket_Desconocido (Sin datos), WorkWeekHrs_is_missing})",0.154385,1.000000,4.862683
2498,"frozenset({WorkRemote_Bucket_Desconocido (Sin datos), WorkWeekHrs_is_missing})","frozenset({Hobbyist_Yes, Neuroticismo_Nivel_Desconocido})",0.154385,0.750726,4.862683
438,frozenset({Neuroticismo_Nivel_Desconocido}),"frozenset({WorkRemote_Bucket_Desconocido (Sin datos), WorkWeekHrs_is_missing})",0.179448,1.000000,4.862683


**Análisis de las reglas obtenidas (sin filtrar):**

Con `min_support=0.15` y `min_confidence=0.6`, Apriori genera **5.324 reglas**. Sin embargo, las 20 con mayor lift están son combinaciones entre `Neuroticismo_Nivel_Desconocido`, `WorkWeekHrs_is_missing`, `CodeRevHrs_is_missing` y `WorkRemote_Bucket_Desconocido (Sin datos)`.

Otras observaciones, como hemos comentado anteriormente son:

1. **Varias reglas alcanzan confianza = 1.000** (ej. `{Neuroticismo_Nivel_Desconocido} → {WorkWeekHrs_is_missing, WorkRemote_Bucket_Desconocido, CodeRevHrs_is_missing}`, fila 2492), por lo que todos los candidatos con neuroticismo desconocido tiene también esas tres variables laborales sin dato.

2. **Muchas filas aparecen en pares con el mismo lift pero distinta confianza** (ej. filas 2503 y 2495, ambas con lift 4.912138). Esto es posible si el lift es simétrico ()`lift(A→B) = lift(B→A)`), porque ambos se calculan como `soporte(A∪B) / (soporte(A) × soporte(B))`. POr el contrario, la confianza no lo es, por eso el mismo itemset genera dos filas con lift idéntico pero confianza distinta según qué lado se tome como antecedente.

3. **Los valores de lift de esta tabla son los más altos de todo el conjunto de reglas** muy por encima de lo que veremos más adelante en las reglas ya tratadas. Esto es porque la presencia de nulos afecta fuertemente a todo el dataset y lo trataremos a lo largo de esta fase.

### 3.3.1. Filtrado de reglas con elementos nulos

Tal cual como hemos comentado antes, vamos a tratar la presencia de valores nulos para evitar que el modelo cometa errores con reglas afectadas. Para ello, haremos el siguiente tratamiento:

Primero identificamos con los sufijos `_is_missing` o `Desconocido` en cualquier ítem de la regla (antecedente o consecuente), y después separamos en un subconjunto de reglas sustantivas: (`reglas_sustantivas`) en vez de borrarlas, así podemos preservar los dos listados:

- El conjunto excluido confirma la consistencia del hallazgo ya reportado en el EDA sobre la estructura no aleatoria del missingness.
- El conjunto `reglas_sustantivas` (5.175 de las 5.324 reglas totales) queda libre de este efecto y es el que se usará en el resto del análisis.

In [4]:
def contiene_item_missing(frozenset_items):
    return any('is_missing' in item or 'Desconocido' in item for item in frozenset_items)

reglas_asociacion['tiene_missing'] = reglas_asociacion.apply(lambda row: contiene_item_missing(row['antecedents']) or contiene_item_missing(row['consequents']),axis=1)
reglas_sustantivas = reglas_asociacion[~reglas_asociacion['tiene_missing']].sort_values('lift', ascending=False)

print(f"Reglas totales: {len(reglas_asociacion)}")
print(f"Reglas con nulos: {reglas_asociacion['tiene_missing'].sum()}")
print(f"Reglas sustantivas restantes: {len(reglas_sustantivas)}")
display(reglas_sustantivas[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(15))

Reglas totales: 5324
Reglas con nulos: 149
Reglas sustantivas restantes: 5175


,antecedents,consequents,support,confidence,lift
4254,"frozenset({Employment_Employed full-time, Amabilidad_Nivel_Alto})","frozenset({CodeRev_Yes, because I see value in code review, MgrIdiot_Very confident})",0.169952,0.720432,3.821839
4251,"frozenset({CodeRev_Yes, because I see value in code review, MgrIdiot_Very confident})","frozenset({Employment_Employed full-time, Amabilidad_Nivel_Alto})",0.169952,0.901584,3.821839
4269,"frozenset({Amabilidad_Nivel_Alto, WelcomeChange_Just as welcome now as I felt last year})","frozenset({Employment_Employed full-time, MgrIdiot_Very confident})",0.152842,0.964118,3.691400
4264,"frozenset({Employment_Employed full-time, Amabilidad_Nivel_Alto, WelcomeChange_Just as welcome now as I felt last year})",frozenset({MgrIdiot_Very confident}),0.152842,1.000000,3.646812
2012,"frozenset({Amabilidad_Nivel_Alto, WelcomeChange_Just as welcome now as I felt last year})",frozenset({MgrIdiot_Very confident}),0.158530,1.000000,3.646812
4228,"frozenset({MgrIdiot_Somewhat confident, WelcomeChange_Just as welcome now as I felt last year})","frozenset({Employment_Employed full-time, Amabilidad_Nivel_Medio})",0.153259,0.733042,3.403841
4229,"frozenset({Employment_Employed full-time, Amabilidad_Nivel_Medio})","frozenset({MgrIdiot_Somewhat confident, WelcomeChange_Just as welcome now as I felt last year})",0.153259,0.711648,3.403841
2003,"frozenset({CodeRev_Yes, because I see value in code review, MgrIdiot_Very confident})",frozenset({Amabilidad_Nivel_Alto}),0.176023,0.933791,3.246911
2007,frozenset({Amabilidad_Nivel_Alto}),"frozenset({CodeRev_Yes, because I see value in code review, MgrIdiot_Very confident})",0.176023,0.612055,3.246911
4247,"frozenset({Employment_Employed full-time, CodeRev_Yes, because I see value in code review, MgrIdiot_Very confident})",frozenset({Amabilidad_Nivel_Alto}),0.169952,0.933259,3.245060


**Análisis del top-15 de reglas sustantivas (antes del filtro de circularidad)**

Una vez hecho el filtro con nulos, vemos que el nuevo top en base al lift está dominado por `Amabilidad_Nivel_Alto` (o `_Medio`) y **sus propias variables constituyentes**: `MgrIdiot_Very confident`, `CodeRev_Yes... because I see value...` y `WelcomeChange_Just as welcome...`. Varias reglas alcanzan de nuevo **confianza = 1.000**. Aquí este patrón no es coincidencia, si no que el algoritmo está redescubiendo (de forma correcta) la fórmula que utilizamos para la construcción del proxy, lo que es matemáticamente esperable. Por ello, se genera un problema de circularidad que hace que quien puntúe algo en `WelcomeChange` y `MgrIdiot` (dos de los tres ingredientes) tienda también a tener `Amabilidad_Nivel_Alto`.

### 3.3.2. Filtrado de reglas de circularidad de los proxies

Para esta celda queda pendiente solucionar el problema de **circularidad**. Como hemos explicado antes, hay reglas que relacionan un proxy de personalidad (`Neuroticismo_Nivel`, `Amabilidad_Nivel`, `Apertura_Nivel`) con alguna de las variables que se usaron para calcularlo en la Libreta 1, las cuales generan reglas de confianza casi perfectas y no aportan nada en cuanto a comportamientos.

**Diccionario de referencia (`INPUTS_BIGFIVE`):** Aquí vamos a aplicar las columnas originales que se promediaron para obtener su `_Score` en `calcular_proxies_bigfive()`:
- `Neuroticismo` ← `ImpSyn`, `JobSat`, `CareerSat`, `WorkWeekHrs_Bin`
- `Amabilidad` ← `WelcomeChange`, `CodeRev`, `MgrIdiot`
- `Apertura` ← `Hobbyist`, `OpenSourcer`, `BetterLife`

En la función `es_regla_circular` lo que hacemos es que para cada regla se recorren todos sus ítems (antecedente y consecuente juntos) en busca de dos señales: 

* Primero vemos qué rasgos aparecen como nivel derivado (ítems que empiezan por `Neuroticismo_Nivel_`, etc.) 
* Segundo, identificamos qué rasgos tienen alguno de sus componentes originales presentes en la regla. Si el mismo rasgo aparece en ambos conjuntos simultáneamente, la regla es circular, ya está combinando el resultado con uno de sus propios ingredientes.

In [5]:
INPUTS_BIGFIVE = {
    'Neuroticismo': ['ImpSyn', 'JobSat', 'CareerSat', 'WorkWeekHrs_Bin'],
    'Amabilidad':   ['WelcomeChange', 'CodeRev', 'MgrIdiot'],
    'Apertura':     ['Hobbyist', 'OpenSourcer', 'BetterLife'],
}

def es_regla_circular(frozenset_items):
    rasgos_presentes = set()
    inputs_presentes = set()
    for item in frozenset_items:
        for rasgo in INPUTS_BIGFIVE:
            if item.startswith(f'{rasgo}_Nivel_'):
                rasgos_presentes.add(rasgo)
        for rasgo, inputs in INPUTS_BIGFIVE.items():
            if any(item.startswith(f'{col}_') for col in inputs):
                inputs_presentes.add(rasgo)
    return len(rasgos_presentes & inputs_presentes) > 0

reglas_sustantivas['es_circular'] = reglas_sustantivas.apply(
    lambda row: es_regla_circular(row['antecedents'] | row['consequents']),
    axis=1
)

reglas_limpias = reglas_sustantivas[~reglas_sustantivas['es_circular']].sort_values('lift', ascending=False)

print(f"Reglas sustantivas previas al filtro: {len(reglas_sustantivas)}")
print(f"Reglas circulares (proxy vs. sus propios componentes): {reglas_sustantivas['es_circular'].sum()}")
print(f"Reglas limpias: {len(reglas_limpias)}")
display(reglas_limpias[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(15))

Reglas sustantivas previas al filtro: 5175
Reglas circulares (proxy vs. sus propios componentes): 491
Reglas limpias: 4684


,antecedents,consequents,support,confidence,lift
1946,"frozenset({CareerSat_Very satisfied, CodeRev_Yes, because I see value in code review})",frozenset({JobSat_Very satisfied}),0.150161,0.626899,2.478835
1952,frozenset({JobSat_Very satisfied}),"frozenset({CareerSat_Very satisfied, WelcomeChange_Just as welcome now as I felt last year})",0.153428,0.606672,2.465174
1950,"frozenset({CareerSat_Very satisfied, WelcomeChange_Just as welcome now as I felt last year})",frozenset({JobSat_Very satisfied}),0.153428,0.623444,2.465174
1948,"frozenset({CodeRev_Yes, because I see value in code review, JobSat_Very satisfied})",frozenset({CareerSat_Very satisfied}),0.150161,0.809460,2.463314
1112,"frozenset({Hobbyist_Yes, JobSat_Very satisfied})",frozenset({CareerSat_Very satisfied}),0.163734,0.805452,2.451118
1469,"frozenset({Employment_Employed full-time, JobSat_Very satisfied})",frozenset({CareerSat_Very satisfied}),0.173748,0.801007,2.437591
1951,"frozenset({JobSat_Very satisfied, WelcomeChange_Just as welcome now as I felt last year})",frozenset({CareerSat_Very satisfied}),0.153428,0.800717,2.436708
1467,"frozenset({CareerSat_Very satisfied, Employment_Employed full-time})",frozenset({JobSat_Very satisfied}),0.173748,0.615498,2.433756
1470,frozenset({JobSat_Very satisfied}),"frozenset({CareerSat_Very satisfied, Employment_Employed full-time})",0.173748,0.687021,2.433756
214,frozenset({JobSat_Very satisfied}),frozenset({CareerSat_Very satisfied}),0.201514,0.796811,2.424820


**Análisis del top-15 de reglas limpias (sin nulos ni circularidad)**

Una vez tenemos un primer listado sin nulos ni circularidad, vemos lo siguiente:

**`JobSat_Very satisfied` ↔ `CareerSat_Very satisfied` domina el bloque** (11 de las 15 reglas), en múltiples combinaciones con `Hobbyist_Yes`, `Employment_Employed full-time`, `CodeRev_Yes... because I see value...` y `WelcomeChange_Just as welcome...`. Este es un resultado coherente, ya que la satisfacción con el puesto actual y la satisfacción con la trayectoria profesional están relacionadas, aunque son preguntas distintas de la encuesta, por lo que la regla no es circular pese a que `JobSat` sí es uno de los inputs de `Neuroticismo_Score`.

**`WorkLoc_Home → JobFactors_Remote work options`** nos dice que todo aquel que ya trabaja desde casa valora explícitamente las opciones de trabajo remoto como factor de decisión al buscar empleo. Es una relación con sentido directo y de valor para el motor de matching, al conectar una circunstancia actual con una preferencia declarada.

### 3.3.3. Reglas que involucran los proxies de Big Five

Despues de excluir las reglas que estaban dominadas por nulos y las circulares por los propios proxies, nos quedan un total de **4.684 reglas limpias**, pero hay un problema, es que en ninguna de las que aparecían en los primeros puestos por lift mencionaba `Neuroticismo_Nivel`, `Amabilidad_Nivel` ni `Apertura_Nivel`. Por ello, tendremos problemas más adelante al comprar las reglas con las de Aurora, que vamos a articular en torno a 3 rasgos: si se ordena solo por lift global, quedan enterradas bajo patrones más genéricos (satisfacción laboral, ubicación de trabajo, etc.).

Por lo tanto, filtramos `reglas_limpias` con el objetivo de aislar el subconjunto donde algún proxy de Big Five aparece en cualquiera de sus niveles (antecedente o consecuente).

In [6]:
def contiene_proxy_bigfive(frozenset_items):
    return any(item.startswith(('Neuroticismo_Nivel_', 'Amabilidad_Nivel_', 'Apertura_Nivel_')) for item in frozenset_items)

reglas_con_proxy = reglas_limpias[
    reglas_limpias.apply(lambda row: contiene_proxy_bigfive(row['antecedents'] | row['consequents']), axis=1)
].sort_values('lift', ascending=False)

print(f"Reglas limpias que sí involucran algún proxy de Big Five: {len(reglas_con_proxy)}")
display(reglas_con_proxy[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(15))

Reglas limpias que sí involucran algún proxy de Big Five: 340


,antecedents,consequents,support,confidence,lift
4617,"frozenset({WorkRemote_Bucket_Presencial / Bajo remoto (<30%), Neuroticismo_Nivel_Medio})","frozenset({Employment_Employed full-time, WorkLoc_Office})",0.193133,0.657968,1.588133
5208,"frozenset({WorkRemote_Bucket_Presencial / Bajo remoto (<30%), Neuroticismo_Nivel_Medio, WelcomeChange_Just as welcome now as I felt last year})",frozenset({WorkLoc_Office}),0.156886,0.703576,1.538779
4614,"frozenset({Employment_Employed full-time, WorkRemote_Bucket_Presencial / Bajo remoto (<30%), Neuroticismo_Nivel_Medio})",frozenset({WorkLoc_Office}),0.193133,0.700953,1.533040
5209,"frozenset({WorkLoc_Office, Neuroticismo_Nivel_Medio})","frozenset({WorkRemote_Bucket_Presencial / Bajo remoto (<30%), WelcomeChange_Just as welcome now as I felt last year})",0.156886,0.693177,1.530094
4616,"frozenset({WorkLoc_Office, Neuroticismo_Nivel_Medio})","frozenset({Employment_Employed full-time, WorkRemote_Bucket_Presencial / Bajo remoto (<30%)})",0.193133,0.853332,1.526992
2216,"frozenset({WorkRemote_Bucket_Presencial / Bajo remoto (<30%), Neuroticismo_Nivel_Medio})",frozenset({WorkLoc_Office}),0.204657,0.697226,1.524889
4613,"frozenset({Employment_Employed full-time, WorkLoc_Office, Neuroticismo_Nivel_Medio})",frozenset({WorkRemote_Bucket_Presencial / Bajo remoto (<30%)}),0.193133,0.919850,1.518397
5207,"frozenset({WorkLoc_Office, Neuroticismo_Nivel_Medio, WelcomeChange_Just as welcome now as I felt last year})",frozenset({WorkRemote_Bucket_Presencial / Bajo remoto (<30%)}),0.156886,0.917100,1.513858
3936,"frozenset({Hobbyist_Yes, WorkRemote_Bucket_Presencial / Bajo remoto (<30%), Neuroticismo_Nivel_Medio})",frozenset({WorkLoc_Office}),0.157426,0.685468,1.499173
4618,"frozenset({Employment_Employed full-time, WorkLoc_Office, Apertura_Nivel_Medio})",frozenset({WorkRemote_Bucket_Presencial / Bajo remoto (<30%)}),0.152391,0.906891,1.497006


**Análisis del top-15 de reglas con proxies de Big Five**

Vemos que **las 15 reglas están dominadas por el nivel `_Medio`** (13 de `Neuroticismo_Nivel_Medio`, 2 de `Apertura_Nivel_Medio`), ninguna menciona `_Alto` ni `_Bajo`. Esto no refleja ausencia de patrones en los extremos, sino que `Neuroticismo_Nivel_Medio` y `Apertura_Nivel_Medio` son las categorías con mayor soporte base dentro de sus rasgos, lo que les facilita superar el umbral `min_support=0.15` en combinación con otras variables, frente a valores `Alto`/`Bajo`.

El patrón dominante conecta `WorkLoc_Office` y `WorkRemote_Bucket_Presencial / Bajo remoto (<30%)` (dos variables que ya consideramos parcialmente redundantes entre sí) con `Neuroticismo_Nivel_Medio`. Los lift (1,46-1,59) son los más bajos vistos hasta ahora en el análisis, coherente con tratarse del nivel intermedio del proxy, el cual representa menos desviación respecto al comportamiento promedio del conjunto de candidatos.

### 3.3.4. Reglas con niveles extremos (Alto/Bajo) de los proxies de Big Five

Antes hemos visto las (`reglas_con_proxy`), pero no aparecían niveles `Alto`/`Bajo` solo nivel `_Medio` en los primeros puestos. Esto nos resulta un problema para los objetivos del proyecto, ya que las reglas de Aurora formula sus condiciones podríamos decir en términos de niveles extremos (alto o bajo, pero no medio), por lo que si no filtramos específicamente, el sesgo del umbral nos ocultaría las reglas más relevantas para comparar más adelante con las reglas del experto.

Ahora filtramos por `reglas_limpias` para quedarnos únicamente con las reglas donde algún proxy de Big Five aparece en su nivel `_Alto` o `_Bajo`, en antecedente o consecuente.


In [7]:
def contiene_extremo_bigfive(frozenset_items):
    return any(item.endswith('_Alto') or item.endswith('_Bajo') 
               for item in frozenset_items 
               if item.startswith(('Neuroticismo_Nivel_', 'Amabilidad_Nivel_', 'Apertura_Nivel_')))

reglas_extremos = reglas_limpias[
    reglas_limpias.apply(lambda row: contiene_extremo_bigfive(row['antecedents'] | row['consequents']), axis=1)].sort_values('lift', ascending=False)

print(f"Reglas con nivel Alto/Bajo de algún proxy: {len(reglas_extremos)}")
display(reglas_extremos[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(15))

Reglas con nivel Alto/Bajo de algún proxy: 87


,antecedents,consequents,support,confidence,lift
5033,"frozenset({Amabilidad_Nivel_Bajo, WorkWeekHrs_Bin_35-45h})","frozenset({Employment_Employed full-time, WorkRemote_Bucket_Presencial / Bajo remoto (<30%)})",0.162844,0.799303,1.430310
5028,"frozenset({Employment_Employed full-time, Amabilidad_Nivel_Bajo, WorkRemote_Bucket_Presencial / Bajo remoto (<30%)})",frozenset({WorkWeekHrs_Bin_35-45h}),0.162844,0.767397,1.407807
5029,"frozenset({Employment_Employed full-time, Amabilidad_Nivel_Bajo, WorkWeekHrs_Bin_35-45h})",frozenset({WorkRemote_Bucket_Presencial / Bajo remoto (<30%)}),0.162844,0.849263,1.401879
5032,"frozenset({Amabilidad_Nivel_Bajo, WorkRemote_Bucket_Presencial / Bajo remoto (<30%)})","frozenset({Employment_Employed full-time, WorkWeekHrs_Bin_35-45h})",0.162844,0.708642,1.399128
2320,"frozenset({Apertura_Nivel_Alto, WorkRemote_Bucket_Presencial / Bajo remoto (<30%)})","frozenset({CodeRev_Yes, because I see value in code review})",0.156920,0.759969,1.393288
5031,"frozenset({Employment_Employed full-time, Amabilidad_Nivel_Bajo})","frozenset({WorkRemote_Bucket_Presencial / Bajo remoto (<30%), WorkWeekHrs_Bin_35-45h})",0.162844,0.609486,1.367146
2456,"frozenset({Amabilidad_Nivel_Bajo, WorkWeekHrs_Bin_35-45h})",frozenset({WorkRemote_Bucket_Presencial / Bajo remoto (<30%)}),0.168735,0.828219,1.367142
2455,"frozenset({Amabilidad_Nivel_Bajo, WorkRemote_Bucket_Presencial / Bajo remoto (<30%)})",frozenset({WorkWeekHrs_Bin_35-45h}),0.168735,0.734278,1.347048
1932,"frozenset({Amabilidad_Nivel_Alto, WorkWeekHrs_Bin_35-45h})",frozenset({Employment_Employed full-time}),0.166460,0.970385,1.337612
5030,"frozenset({Amabilidad_Nivel_Bajo, WorkRemote_Bucket_Presencial / Bajo remoto (<30%), WorkWeekHrs_Bin_35-45h})",frozenset({Employment_Employed full-time}),0.162844,0.965087,1.330308


Tras observar los resultados vemos que destacan tres patrones principalmente:

**1. `Amabilidad_Nivel_Bajo` domina el ranking, combinada siempre con el mismo trío de variables:** `WorkWeekHrs_Bin_35-45h`, `WorkRemote_Bucket_Presencial / Bajo remoto (<30%)` y `Employment_Employed full-time`. Diez de las quince reglas giran en torno a esta combinación, en distintas direcciones. El patrón describe un perfil concreto, donde los candidatos con baja amabilidad tienden a concentrarse en el modelo de trabajo más convencional: jornada completa, horario estándar (35-45h) y predominantemente presencial, más que en modalidades flexibles o remotas. Los lift son moderados (1,3-1,4) son coherente con la naturaleza indirecta del proxy.

**2. `Apertura_Nivel_Alto` se asocia con valorar la revisión de código por convicción propia**, no por obligación (`{Apertura_Nivel_Alto, ...} → {CodeRev_Yes, because I see value in code review}`, lift ≈1,33-1,39). Tiene sentido, ya que los candidatos ganan experiencia mediante la curiosidad, receptividad y retroalimentación, y el propio `CodeRev` fue uno de los inputs usados para construir `Amabilidad_Score`, así que esta relación es una asociación entre dos rasgos distintos.

**3. La regla más relevante para el objetivo del proyecto es la de la fila 1903:** `{Neuroticismo_Nivel_Alto, Employment_Employed full-time} → {WorkRemote_Bucket_Presencial / Bajo remoto (<30%)}` (lift 1,32). Es la misma relación que ya se había identificado antes, y merece destacarse porque **contradice la dirección intuitiva de las reglas de Aurora**, que asocia el trabajo remoto alto (>70%) con mayor riesgo de aislamiento y fatiga digital, sugiriendo que el remoto alto sería más problemático para perfiles con mayor inestabilidad emocional. 

Los datos pre-pandemia muestran lo contrario: los candidatos con neuroticismo alto tienden a trabajar de forma **presencial**.

### 3.4. Generación de Itemsets Frecuentes — 2º Algoritmo --> FP-Growth

**FP-Growth** (*Frequent Pattern Growth*) es el segundo algoritmo clásico de minería de reglas de asociación que vamos a emplear en este proyecto. Este algotirmo propuesto en el 2000 permite extraer reglas de asociacion a partir de itemsets frecuentes, pero a diferencia de Apriori, estos se identifican sin necesidad de generar candidatos para cada tamaño, si no que genera y poda combinaciones candidatas de forma iterativa.

En terminos generales, el algoritmo emplea una estructura de árbol (Frequent Pattern tree) donde almacena toda la información de las transacciones. Esta estructura permite comprimir información, haciendo posible que sea más eficiente y dividirlo en datos condicionales. Finalmente, cada partición se analiza de forma separada y se concatenan los resultados obtenidos.

Para los parámetros empleados, vamos a mantener exactamente los mismos que Apriori (`min_support=0.15`, `max_len=4`) para que la comparación entre ambos sea válida. A diferencia de apriori, aquí no disponemos de un parámetro low_memory`, ya que su propia estructura en árbol es, por diseño, más compacta en memoria que la generación de candidatos de Apriori.

In [8]:
start = time.time()
frequent_itemsets_fpgrowth = fpgrowth( df_matriz,min_support=0.15, use_colnames=True,max_len=4)
tiempo_fpgrowth = time.time() - start

print(f"Itemsets frecuentes de FP-Growth: {len(frequent_itemsets_fpgrowth)}")
print(f"Tiempo de ejecución: {tiempo_fpgrowth:.2f} segundos")

Itemsets frecuentes de FP-Growth: 2188
Tiempo de ejecución: 22.68 segundos


**Verificación de equivalencia entre algoritmos**

Ahora que tenemos los dos algoritmos probados vamos a validar que estos, bajo la misma configuración y umbrales sean equivalentes. En caso contrario, destacaríamos aquel que nos ofreciera mejores resultados

In [9]:
reglas_fpgrowth = association_rules(frequent_itemsets_fpgrowth, metric="confidence",min_threshold=0.6)

print(f"Itemsets — Apriori: {len(itemsets_frecuentes)} | FP-Growth: {len(frequent_itemsets_fpgrowth)}")
print(f"Reglas — Apriori: {len(reglas_asociacion)} | FP-Growth: {len(reglas_fpgrowth)}")

Itemsets — Apriori: 2188 | FP-Growth: 2188
Reglas — Apriori: 5324 | FP-Growth: 5324


Si observamos los resultados vemos que Apriori y FP-Growth son algoritmos que nos ofrecen los mismos resultados prácticamente, por lo que cualquiera de los dos nos serían útiles, pero tenemos que comparar cuales de ellos sería más eficiente.



### 3.5. Comparativa de rendimiento: Apriori vs. FP-Growth

Como hemos podido comprobar, tanto Apriori como FP-Growth generan exactamente los mismos itemsets y mismo número de reglas con la misma configuración, por lo que ahora queda la pregunta de cual es más eficiente computacionalmente hablando, en tiempos de ejecución.
Para ello, vamos a volver a ejecutar los dos algortimos con los mismos hiperparámetros y esta vez contamos los tiempos, para ver cual de ellos es más rápido y la diferencia real entre ambas en proporción.



In [10]:
start = time.time()
frequent_itemsets_apriori_tiempos = apriori(df_matriz, min_support=0.15, use_colnames=True, max_len=4, low_memory=True)
tiempo_apriori = time.time() - start

print(f"Tiempo Apriori:   {tiempo_apriori:.2f}s")
print(f"Tiempo FP-Growth: {tiempo_fpgrowth:.2f}s")
print(f"FP-Growth es {tiempo_apriori/tiempo_fpgrowth:.1f}x más rápido" if tiempo_fpgrowth < tiempo_apriori else f"Apriori es {tiempo_fpgrowth/tiempo_apriori:.1f}x más rápido")

Tiempo Apriori:   10.56s
Tiempo FP-Growth: 22.68s
Apriori es 2.1x más rápido


### 3.6. Tercer algoritmo --> Eclat

El tercer algoritmo es **Eclat** (*Equivalence Class Transformation*). Creado en el 2000 por Zaki, su objetivo es encontrar patrones frecuentes. La principal diferencia entre este algoritmo y los anteriores, aquí el dataset no se recorre fila a fila, si no que usa una representación vertical en la que cada item mantiene la lista de identificadores de transacción. Posteriormente, con esa lista de ids realiza un cálculo del soporte mediante la **intersección** de los tidsets de sus componentes. En teoría, esta es una operación muy eficiente, lo que hace de Eclat sea algoritmo atractivo por su complejidad.

Como Eclat no está implementado en `mlxtend`, para este caso recurrimos a `pyECLAT`. Esta librería es la alternativa más extendida para Python, y a diferencia de los dos algoritmos anteriores, `pyECLAT` espera el dataset en formato de **lista de transacciones**, donde cada fila contiene los nombres de los ítems presentes.

**¿Por qué se trabaja con una muestra de 10.000 filas y no con el dataset completo?** Realizando diversas pruebas sobre todas las filas completas con `max_combination=4` (el mismo alcance usado en Apriori y FP-Growth), vemos que la ejecución no se completa: tras más de 50 minutos, la combinación de 3 ítems no termina. POr ello, como alternativa evaulamos el algoritmo sobre una muestra reducida y un alcance menor, lo que es suficiente para poder obtener una medida de rendimiento comparable pudiendo completar el resto de la libreta. Debido a esto, también he cambiado el `max_combination=2`

Una cosa a tener en cuenta es que `pyECLAT` asume que el dataframe que se pasa como entrada tiene un índice contiguo que empieza con 0. Por ello,al tomar una muestra los índices originales que no son contiguos se conserva y provoca un fallo al construir la tabla binaria, por ello hay que incluir un `.reset_index(drop=True)`

Con los resultados que obtengamos aquí vamos a interpretar y comparar con el resto de algoritmos anteriores.

**Transformación a formato de lista de transacciones**

como `pyECLAT` no acepta la matriz binaria `df_matriz` como entrada, es necesario transformarla a una **lista de los nombres de los ítems presentes**, sin los ítems ausentes. Este formato clásico sería por ejemplo: `["Hobbyist_Yes", "Employment_Employed full-time", ...]`.

La función `matriz_a_transacciones` lo que hace es recorrer cada fila de `df_matriz` y, usando el propio booleano de cada columna como máscara (`row.index[row]`), de ellas extrae únicamente los nombres de las columnas activas (`True`) y descarta el resto.

In [11]:
def matriz_a_transacciones(df_bin):
    return df_bin.apply(lambda row: row.index[row].tolist(), axis=1)

transacciones = matriz_a_transacciones(df_matriz)
df_transacciones = pd.DataFrame(transacciones.tolist())

print(f"Forma del dataset de transacciones: {df_transacciones.shape}")
display(df_transacciones.head(3))

Forma del dataset de transacciones: (88778, 34)


,0,1,2,3,4,5,6,7,8,9,...,24,25,26,27,28,29,30,31,32,33
0,WorkWeekHrs_is_missing,CodeRevHrs_is_missing,Hobbyist_Yes,OpenSourcer_Never,"Employment_Not employed, and not looking for work",BetterLife_Yes,SocialMedia_Twitter,Extraversion_Online,SOVisitTo_Find answers to specific questions;Learn how to do things I didn’t necessarily look for,WelcomeChange_Just as welcome now as I felt last year,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,WorkWeekHrs_is_missing,CodeRevHrs_is_missing,"JobFactors_Languages, frameworks, and other technologies I'd be working with",JobFactors_Financial performance or funding status of the company or organization,JobFactors_Specific department or team I'd be working on,Hobbyist_No,OpenSourcer_Less than once per year,"Employment_Not employed, but looking for work",JobSeek_I am actively looking for a job,LastHireDate_I've never had a job,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,CodeRevHrs_is_missing,WorkChallenge_Distracting work environment,WorkChallenge_Lack of support from management,WorkChallenge_Inadequate access to necessary tools,"JobFactors_Languages, frameworks, and other technologies I'd be working with",JobFactors_Flex time or a flexible schedule,JobFactors_Remote work options,Hobbyist_Yes,OpenSourcer_Never,Employment_Employed full-time,...,SocialMedia_Reddit,Extraversion_In real life (in person),SOVisitTo_Find answers to specific questions;Learn how to do things I didn’t necessarily look for,SOPartFreq_Less than once per month or monthly,WelcomeChange_Just as welcome now as I felt last year,WorkRemote_Bucket_Presencial / Bajo remoto (<30%),Neuroticismo_Nivel_Medio,Amabilidad_Nivel_Bajo,Apertura_Nivel_Medio,WorkWeekHrs_Bin_35-45h


Como cada candidato tiene un número distinto de items activos (por ej. alguien con muchos "Not Applicable" ya excluidos en la Libreta 2 tendrá menos ítems marcados que alguien con el perfil completo) al convertir la lista de listas en un dataframe, pandas iguala todas las filas a la longitud más larga y rellena con `None` los huecos sobrantes. Por lo tanto, el número de columnas (34) corresponde al **máximo de ítems activos que tiene un único candidato** en todo el dataset, a diferencia de las columnas de `df_matriz`.


In [12]:
muestra = df_transacciones.sample(n=10000, random_state=42).reset_index(drop=True)
eclat_instance = ECLAT(data=muestra, verbose=True)

start = time.time()
itemsets_eclat, reglas_eclat_raw = eclat_instance.fit(min_support=0.15, min_combination=1,max_combination=2, separator=' & ')
tiempo_eclat_muestra = time.time() - start

print(f"Tiempo ECLAT para 10k filas, hasta combinaciones de 2: {tiempo_eclat_muestra:.2f}s")
print(f"Itemsets encontrados: {len(itemsets_eclat) if itemsets_eclat is not None else 0}")

100%|██████████| 204/204 [00:00<00:00, 1489.55it/s]


Combination 1 by 1


82it [00:06, 13.56it/s]


Combination 2 by 2


3321it [02:27, 22.58it/s]

Tiempo ECLAT para 10k filas, hasta combinaciones de 2: 153.23s
Itemsets encontrados: 622


### 3.6.1. Resumen comparativo de los tres algoritmos

| Algoritmo | Filas procesadas | Alcance combinación | Tiempo | Itemsets frecuentes |
|---|---|---|---|---|
| Apriori (`low_memory=True`) | 88.778 (completo) | hasta 4 | 10,90s | 2.188 |
| FP-Growth | 88.778 (completo) | hasta 4 | 20,75s | 2.188 |
| Eclat (`pyECLAT`) | 10.000 (muestra) | hasta 2 | 153,23s | 622 |

**Conclusiones del bloque de comparación de algoritmos:**

1. **Apriori y FP-Growth son equivalentes en resultado** (2.188 itemsets, 5.324 reglas idénticas bajo los mismos umbrales), confirmando empíricamente que ambos algoritmos resuelven el mismo problema matemático y solo difieren en la estrategia de recorrido del espacio de búsqueda.

2. **Apriori resultó más rápido que FP-Growth en esta ejecución concreta** (10,90s frente a 20,75s, 1,9x). Esto puede ser por el uso de `low_memory=True` en Apriori (necesario tras agotar la memoria disponible con un `min_support` más bajo en una prueba inicial) combinado con el `min_support=0.15` empleado, que redujo considerablemente el espacio de búsqueda real de Apriori, mientras que FP-Growth incurre en un coste fijo de construcción del FP-Tree sobre todo el dataset que, para este tamaño y umbral, no llegó a compensarse.

3. **Eclat (vía `pyECLAT`) resultó sustancialmente más lento en la práctica**, esto evidencia una brecha entre la complejidad algorítmica teórica de Eclat y la eficiencia real de esta implementación concreta en Python, que recurre a enumeración combinatoria sin vectorización.

En conjunto, estos resultados justifican la elección de **Apriori** como algoritmo principal para el resto del proyecto (dado su buen rendimiento y la posibilidad de operar sobre el dataset completo sin muestreo), reservando FP-Growth como validación de equivalencia y Eclat como referencia comparativa de rendimiento, documentada pero no explotada en profundidad dado su coste computacional en esta implementación.

### 3.7. Exportación de la "Base de Conocimiento"

Las reglas generadas por Apriori (`reglas_asociacion`) y su versión depurada tras el filtrado de nulos y circularidad (`reglas_limpias`) las exportamos ahora a CSV para poder utilizarlas así en fases posteriores del proyecto, donde compararemos más adelante con las reglas expertas de Aurora, para la cual he decidio hacer en una libreta aparte e independiente.

El motivo por que cual exportamos los dos conjuntos es por transparencia, las `reglas_asociacion` (5.324 reglas) como tal es el resultado en bruto y las `reglas_limpias` (4.684 reglas) son aquellas que vamos a comparar en un futuro ya que no tienen problemas con los nulos y problemas circulares por los proxies del Big Five.

En cuanto a la exportación, recordamos que las columnas `antecedents` y `consequents` de `mlxtend` son objetos `frozenset`, los cuales no tienen una representación legible ni reconstruible en un fichero CSV plano. Por ello, antes de exportar aqui convertimos en cadenas de texto con los ítems ordenados alfabéticamente y separados por comas (ej. `frozenset({'A', 'B'})` → `"A, B"`), así el fichero de salida será legible y permitirá realizar comparaciones a futuro.

In [13]:
def exportar_reglas(df_reglas, path):
    df_export = df_reglas.copy()
    df_export['antecedents'] = df_export['antecedents'].apply(lambda x:', '.join(sorted(x)))
    df_export['consequents'] = df_export['consequents'].apply(lambda x:', '.join(sorted(x)))
    df_export.to_csv(path, index=False)
    print(f"Guardado en {path} — {len(df_export)} reglas")

exportar_reglas(reglas_asociacion, config.REGLAS_APRIORI_PATH)
exportar_reglas(reglas_limpias, config.REGLAS_LIMPIAS_PATH)

Guardado en /run/media/miguel-angel/Datos compartidos/TFM/data/processed/reglas_apriori_soft.csv — 5324 reglas
Guardado en /run/media/miguel-angel/Datos compartidos/TFM/data/processed/reglas_limpias_soft.csv — 4684 reglas
